# 07 - RAG Document Q&A

## Stage 8 Objective

Build retrieval-augmented Q&A over the Hugging Face-built LEDGAR clauses using `sentence-transformers/all-MiniLM-L6-v2` and `faiss-cpu`.

## Approach

1. Load `data/processed/classification_dataset.csv` from Notebook 03.
2. Build sentence-transformer embeddings for each clause.
3. Create a FAISS inner-product index over normalized embeddings.
4. Retrieve top-k relevant clauses for a user question.
5. Generate a simple answer using only the retrieved clauses.

If the embedding model or FAISS is unavailable, the notebook falls back to lexical retrieval so the flow remains executable.

## 1. Configure Paths and Imports

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rag_qa import (
    DEFAULT_EMBEDDING_MODEL,
    answer_question,
    build_faiss_index,
    embed_texts,
    load_embedding_model,
    retrieve_top_k,
)

CLASSIFICATION_DATASET_PATH = PROJECT_ROOT / 'data' / 'processed' / 'classification_dataset.csv'
TOP_K = 3

print(f'Classification dataset path: {CLASSIFICATION_DATASET_PATH}')
print(f'Embedding model: {DEFAULT_EMBEDDING_MODEL}')

## 2. Load Hugging Face Clauses

Notebook 03 creates `classification_dataset.csv` from the public LEDGAR dataset.

In [ ]:
required_columns = ['clause_id', 'document_id', 'clause_number', 'clause_text']
source_columns = ['clause_id', 'clause_text', 'clause_type']

if not CLASSIFICATION_DATASET_PATH.exists():
    raise FileNotFoundError(f'Missing dataset: {CLASSIFICATION_DATASET_PATH}. Run Notebook 03 first.')

source_df = pd.read_csv(CLASSIFICATION_DATASET_PATH)
for column in source_columns:
    if column not in source_df.columns:
        source_df[column] = ''
source_df['clause_text'] = source_df['clause_text'].fillna('').astype(str).str.strip()
source_df = source_df[source_df['clause_text'] != ''].reset_index(drop=True)

clauses_df = pd.DataFrame(
    {
        'clause_id': source_df['clause_id'],
        'document_id': 'huggingface_ledgar',
        'clause_number': range(1, len(source_df) + 1),
        'clause_text': source_df['clause_text'],
    }
)

if clauses_df.empty:
    raise ValueError('No clauses found. Re-run Notebook 03 and check Hugging Face dataset access.')

clauses = clauses_df[required_columns].to_dict(orient='records')
print(f'Loaded {len(clauses)} Hugging Face clause(s).')
display(clauses_df[required_columns].head(10))

## 3. Build Embeddings and FAISS Index

This uses normalized sentence-transformer embeddings with an inner-product FAISS index. For normalized vectors, inner product is equivalent to cosine similarity.

In [ ]:
embedding_model = None
embeddings = None
faiss_index = None
retrieval_mode = 'lexical_fallback'

try:
    embedding_model = load_embedding_model(DEFAULT_EMBEDDING_MODEL)
    embeddings = embed_texts(embedding_model, clauses_df['clause_text'].tolist())
    faiss_index = build_faiss_index(embeddings)
    retrieval_mode = 'sentence_transformers_faiss'
except Exception as exc:
    print(f'Falling back to lexical retrieval because embedding/FAISS setup failed: {exc}')

print(f'Retrieval mode: {retrieval_mode}')
if embeddings is not None:
    print(f'Embedding shape: {embeddings.shape}')

## 4. Retrieve Top-K Clauses for a Question

In [ ]:
question = 'What happens if rent payment is late?'

retrieved = retrieve_top_k(
    question,
    clauses,
    model=embedding_model,
    index=faiss_index,
    embeddings=embeddings,
    top_k=TOP_K,
)

retrieved_df = pd.DataFrame(retrieved)
print(f'Question: {question}')
display(retrieved_df)

## 5. Generate a Grounded Rule-Based Answer

No generation model is required here. The answer uses only the retrieved clauses and includes the educational-use disclaimer.

In [ ]:
qa_result = answer_question(
    question,
    clauses,
    model=embedding_model,
    index=faiss_index,
    embeddings=embeddings,
    top_k=TOP_K,
)

print(qa_result['answer'])
display(pd.DataFrame(qa_result['retrieved_clauses']))

## 6. Try More Questions

In [ ]:
sample_questions = [
    'Can either party terminate the agreement?',
    'Is the landlord liable for indirect damages?',
    'Does the tenant need to keep information confidential?',
]

for sample_question in sample_questions:
    result = answer_question(
        sample_question,
        clauses,
        model=embedding_model,
        index=faiss_index,
        embeddings=embeddings,
        top_k=TOP_K,
    )
    print('\nQuestion:', sample_question)
    print('Answer:', result['answer'])
    for clause in result['retrieved_clauses']:
        print(f"- Clause {clause['clause_number']} | score={clause['score']:.4f} | {clause['clause_text']}")